In [161]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, roc_curve, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import shap
import seaborn as sns
import warnings
import math
warnings.filterwarnings('ignore')

In [162]:
from causalnex.structure.notears import from_pandas

In [163]:
from sklearn.preprocessing import MinMaxScaler

In [164]:
df = pd.read_excel("Telco-Customer-Churn.xlsx")

In [165]:
import pickle
import copy
from causalnex.network import BayesianNetwork
from causalnex.structure import StructureModel
from causalnex.plots import plot_structure
from IPython.display import Image
from sklearn.metrics import mutual_info_score
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE

In [166]:
import json

In [167]:
import networkx as nx

In [168]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        7043 non-null   object 
 1   Gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   Tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [9]:
df.describe()

,SeniorCitizen,Tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [10]:
# TOTAL CHARGES TO NUMERIC
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [11]:
# MonthlyCharges TO NUMERIC
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df['MonthlyCharges'].fillna(df['MonthlyCharges'].median(), inplace=True)

In [12]:
# Tenure TO NUMERIC
df['Tenure'] = pd.to_numeric(df['Tenure'], errors='coerce')
df['Tenure'].fillna(df['Tenure'].median(), inplace=True)

In [13]:
# DROP CUSTOMER ID
# df.drop(['CustomerID'], axis=1, inplace=True)

In [14]:
# DEFINE CATEGORICAL AND NUMERICAL
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [15]:
print (categorical_cols)

Index(['CustomerID', 'Gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='object')


In [154]:
print (numerical_cols)

Index(['SeniorCitizen', 'Tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')


In [155]:
df[categorical_cols].nunique()  

CustomerID          7043
Gender                 2
Partner                2
Dependents             2
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
Churn                  2
dtype: int64

In [156]:
# RENAME CATEGORIES TO AVOID SPACES
for columna in categorical_cols:
    df[columna] = df[columna].astype(str).str.replace(' ', '_')

In [157]:
# ENCODE TARGET VARIABLE
df['Churn'] = df['Churn'].replace({'No': 0, 'Yes': 1})

In [158]:
# -----------------------------
# Binary categorical columns
# -----------------------------
binary_cols = [
    'Gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

# -----------------------------
# Multi-category columns
# -----------------------------
multi_cat_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

# -----------------------------
# Numerical columns
# SeniorCitizen is binary (0,1)
# -----------------------------
numerical_cols = [
    'Tenure',
    'MonthlyCharges',
    'TotalCharges'
]

In [159]:
binary_mappings = {
    'Gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1}
}

In [160]:
for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping)


In [161]:
# ============================================
# OneHotEncoding for multi-category variables
# ============================================

ohe = OneHotEncoder(
    sparse_output=False,
    drop=None  # keep all categories for causal discovery
)

In [162]:
encoded_cat = ohe.fit_transform(df[multi_cat_cols])

In [163]:
print (encoded_cat)

[[0. 1. 0. ... 0. 1. 0.]
 [1. 0. 0. ... 0. 0. 1.]
 [1. 0. 0. ... 0. 0. 1.]
 ...
 [0. 1. 0. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 0. 1.]
 [1. 0. 0. ... 0. 0. 0.]]


In [164]:
#std = StandardScaler()

In [165]:
#encoded_num = std.fit_transform(df[numerical_cols])

In [168]:
minmax = MinMaxScaler(feature_range=(0, 1))

In [169]:
encoded_num = minmax.fit_transform(df[numerical_cols])

In [170]:
print (encoded_num)

[[0.01388889 0.11542289 0.0012751 ]
 [0.47222222 0.38507463 0.21586661]
 [0.02777778 0.35422886 0.01031041]
 ...
 [0.15277778 0.11293532 0.03780868]
 [0.05555556 0.55870647 0.03321025]
 [0.91666667 0.86965174 0.78764136]]


In [171]:
# Get encoded categorical feature names
columns = ohe.get_feature_names_out(multi_cat_cols)

In [172]:
print (columns)

['MultipleLines_No' 'MultipleLines_No_phone_service' 'MultipleLines_Yes'
 'InternetService_DSL' 'InternetService_Fiber_optic' 'InternetService_No'
 'OnlineSecurity_No' 'OnlineSecurity_No_internet_service'
 'OnlineSecurity_Yes' 'OnlineBackup_No' 'OnlineBackup_No_internet_service'
 'OnlineBackup_Yes' 'DeviceProtection_No'
 'DeviceProtection_No_internet_service' 'DeviceProtection_Yes'
 'TechSupport_No' 'TechSupport_No_internet_service' 'TechSupport_Yes'
 'StreamingTV_No' 'StreamingTV_No_internet_service' 'StreamingTV_Yes'
 'StreamingMovies_No' 'StreamingMovies_No_internet_service'
 'StreamingMovies_Yes' 'Contract_Month-to-month' 'Contract_One_year'
 'Contract_Two_year' 'PaymentMethod_Bank_transfer_(automatic)'
 'PaymentMethod_Credit_card_(automatic)' 'PaymentMethod_Electronic_check'
 'PaymentMethod_Mailed_check']


In [173]:
encoded_cat_df = pd.DataFrame(
    encoded_cat,
    columns=ohe.get_feature_names_out(multi_cat_cols),
    index=df.index
)

In [174]:
encoded_num_df = pd.DataFrame(
    encoded_num,
    columns=numerical_cols,
    index=df.index
)

In [175]:
# ============================================
# Final dataset
# ============================================

df_transformed = pd.concat(
    [
        df[binary_cols],
        encoded_num_df,
        encoded_cat_df,
        df[['SeniorCitizen','CustomerID','Churn']]
    ],
    axis=1
)

In [176]:
print (df_transformed.columns)

Index(['Gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
       'Tenure', 'MonthlyCharges', 'TotalCharges', 'MultipleLines_No',
       'MultipleLines_No_phone_service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber_optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No_internet_service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No_internet_service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No_internet_service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No_internet_service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No_internet_service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No_internet_service',
       'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One_year',
       'Contract_Two_year', 'PaymentMethod_Bank_transfer_(automatic)',
       'PaymentMethod_Credit_card_(automatic)',
     

In [177]:
df_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 42 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Gender                                   7043 non-null   int64  
 1   Partner                                  7043 non-null   int64  
 2   Dependents                               7043 non-null   int64  
 3   PhoneService                             7043 non-null   int64  
 4   PaperlessBilling                         7043 non-null   int64  
 5   Tenure                                   7043 non-null   float64
 6   MonthlyCharges                           7043 non-null   float64
 7   TotalCharges                             7043 non-null   float64
 8   MultipleLines_No                         7043 non-null   float64
 9   MultipleLines_No_phone_service           7043 non-null   float64
 10  MultipleLines_Yes                        7043 no

In [178]:
df_transformed.describe()

,Gender,Partner,Dependents,PhoneService,PaperlessBilling,Tenure,MonthlyCharges,TotalCharges,MultipleLines_No,MultipleLines_No_phone_service,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One_year,Contract_Two_year,PaymentMethod_Bank_transfer_(automatic),PaymentMethod_Credit_card_(automatic),PaymentMethod_Electronic_check,PaymentMethod_Mailed_check,SeniorCitizen,Churn
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,...,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.504756,0.483033,0.299588,0.903166,0.592219,0.449599,0.462803,0.261149,0.481329,0.096834,...,0.387903,0.550192,0.209144,0.240664,0.219225,0.216101,0.335794,0.228880,0.162147,0.265370
std,0.500013,0.499748,0.458110,0.295752,0.491457,0.341104,0.299403,0.261397,0.499687,0.295752,...,0.487307,0.497510,0.406726,0.427517,0.413751,0.411613,0.472301,0.420141,0.368612,0.441561
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,0.125000,0.171642,0.044245,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,1.000000,0.402778,0.518408,0.159090,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,0.763889,0.712438,0.434780,1.000000,0.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [179]:
print(df_transformed.head(10))

   Gender  Partner  Dependents  PhoneService  PaperlessBilling    Tenure  \
0       0        1           0             0                 1  0.013889   
1       1        0           0             1                 0  0.472222   
2       1        0           0             1                 1  0.027778   
3       1        0           0             0                 0  0.625000   
4       0        0           0             1                 1  0.027778   
5       0        0           0             1                 1  0.111111   
6       1        0           1             1                 1  0.305556   
7       0        0           0             0                 0  0.138889   
8       0        1           0             1                 1  0.388889   
9       1        0           1             1                 0  0.861111   

   MonthlyCharges  TotalCharges  MultipleLines_No  \
0        0.115423      0.001275               0.0   
1        0.385075      0.215867               1.0   
2   

In [180]:
#df_transformed.to_excel("Telco-Churn-OneHot-MinMax.xlsx", index=False)

In [169]:
#START
df_transformed = pd.read_excel("Telco-Churn-OneHot-MinMax.xlsx")

In [170]:
# DROP CUSTOMER ID
df_transformed.drop(['CustomerID'], axis=1, inplace=True)

In [171]:
df_transformed = df_transformed.astype(float)

In [172]:
# NO TEARS
# learn
#sm = from_pandas(df_transformed)

In [173]:
#with open("model_notears.pkl", "wb") as f:
#     pickle.dump(sm, f)

In [174]:
with open("model_notears.pkl", "rb") as f:
    restored_model = pickle.load(f)

In [175]:
sm = restored_model

In [176]:
num_nodes = len(sm.nodes)
print(f"Number of nodes: {num_nodes}")

Number of nodes: 41


In [177]:
weights = []
all_edges = []

for u, v, data in sm.edges(data=True):
    w = (data['weight'])
    weights.append(w)
    all_edges.append({"From": u, "To": v, "Weight": round(w, 3)})

In [178]:
stats = {
    "Statistic": ["Count", "Mean", "Median", "Std Dev", "Min", "Max"],
    "Value": [
        len(weights),
        round(pd.Series(weights).mean(), 3),
        round(pd.Series(weights).median(), 3),
        round(pd.Series(weights).std(), 3),
        round(pd.Series(weights).min(), 3),
        round(pd.Series(weights).max(), 3),
    ]
}
df_stats = pd.DataFrame(stats)

In [179]:
print(df_stats)

  Statistic     Value
0     Count  1640.000
1      Mean     0.024
2    Median     0.000
3   Std Dev     0.600
4       Min    -5.494
5       Max    15.694


In [180]:
sm_reduced = copy.deepcopy(restored_model)

In [181]:
threshold = 0.078
edges_to_remove = [(u, v) for u, v, w in sm_reduced.edges(data=True) if abs(w["weight"]) < threshold]
sm_reduced.remove_edges_from(edges_to_remove)

In [182]:
weights = []
reduced_edges = []

for u, v, data in sm_reduced.edges(data=True):
    w = data["weight"]
    weights.append(w)
    reduced_edges.append({"From": u, "To": v, "Weight": round(w, 3)})

In [183]:
stats = {
    "Statistic": ["Count", "Mean", "Median", "Std Dev", "Min", "Max"],
    "Value": [
        len(weights),
        round(pd.Series(weights).mean(), 3),
        round(pd.Series(weights).median(), 3),
        round(pd.Series(weights).std(), 3),
        round(pd.Series(weights).min(), 3),
        round(pd.Series(weights).max(), 3),
    ]
}
df_stats = pd.DataFrame(stats)

In [184]:
print (df_stats)

  Statistic    Value
0     Count  297.000
1      Mean    0.120
2    Median    0.109
3   Std Dev    1.407
4       Min   -5.494
5       Max   15.694


In [185]:
all_edge_attributes = {
    "color":{
        "color": "#FFFFFFD9",
    },
    "font": {
            "color": 'green',
            "face" : 'arial',
            "background": '#FFFFFFD9',
            "strokeWidth": 1 ,
            "strokeColor": 'green',
            "bold": True,
            "size": 30
        },
    "length": 300,
    "label" : ""
}

In [186]:
edge_attributes = {};
for u, v, data in sm_reduced.edges(data=True):
    weight = data["weight"]
    edge_attributes[(u, v)] = {
        "color": {
            "color": "black",
            "highlight": "#79d279"
        },
        "length": 500,
        "shadow": {
            "enabled": True,
            "color": "gray"
        },
        "label": ""  # Using the edge weight as the label
    }

In [187]:
all_node_attributes = {
    "font": {
        "color": "blue",
        "face": "Helvetica",
        "size": 40
    }
}

In [188]:
plot_options = {
    "bgcolor": "white"
}

In [189]:
viz = plot_structure(
    sm_reduced, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
#viz.toggle_physics(False)
viz.show("supporting_files/fully_connected_notears.html")

supporting_files/fully_connected_notears.html


In [190]:
center_node = "Churn"
subgraph = nx.ego_graph(sm_reduced, center_node, radius=1, center=True, undirected = True)

In [191]:
print(type(subgraph))
print(isinstance(subgraph, StructureModel))

<class 'causalnex.structure.structuremodel.StructureModel'>
True


In [193]:
# Remove all isolated nodes
isolated_nodes = list(nx.isolates(subgraph))
subgraph.remove_nodes_from(isolated_nodes)

In [194]:
all_edges = [(u, v, w) for u, v, w in subgraph.edges(data=True)]

In [195]:
print (all_edges)

[('MultipleLines_No_phone_service', 'TechSupport_No', {'origin': 'learned', 'weight': 0.3440329759709901}), ('MultipleLines_No_phone_service', 'Churn', {'origin': 'learned', 'weight': 0.1658184365448262}), ('Tenure', 'MultipleLines_No_phone_service', {'origin': 'learned', 'weight': 0.4115119963750835}), ('Tenure', 'Contract_One_year', {'origin': 'learned', 'weight': 0.6308009655310122}), ('Tenure', 'Churn', {'origin': 'learned', 'weight': -0.12732708108281068}), ('InternetService_No', 'Tenure', {'origin': 'learned', 'weight': 0.2347637157738336}), ('InternetService_No', 'TechSupport_No', {'origin': 'learned', 'weight': -0.35160511701425623}), ('InternetService_No', 'Churn', {'origin': 'learned', 'weight': 0.10074848763277956}), ('TechSupport_No', 'Churn', {'origin': 'learned', 'weight': 0.08126579509329547}), ('MonthlyCharges', 'TotalCharges', {'origin': 'learned', 'weight': 0.5165438933643935}), ('MonthlyCharges', 'Churn', {'origin': 'learned', 'weight': 0.5620013368540109}), ('TotalC

In [196]:
edge_attributes = {};
for u, v, data in subgraph.edges(data=True):
    weight = data["weight"]
    edge_attributes[(u, v)] = {
        "color": {
            "color": "black",
            "highlight": "#79d279"
        },
        "length": 500,
        "shadow": {
            "enabled": True,
            "color": "gray"
        },
        "label": f"{weight:.3f}"  # Using the edge weight as the label
    }


In [197]:
opt = {
    "physics": {
        "enabled":False,
        "solver": "barnesHut",
        "barnesHut": {
            "gravitationalConstant": -2000,
            "centralGravity": 0.3,
            "springLength": 100,
            "springConstant": 0.04,
            "damping": 0.09,
            "avoidOverlap": 1
        }
    },
    "edges": {
        "smooth": {
            "enabled": True,
            "type": "cubicBezier",
            "forceDirection": "horizontal",
            "roundness": 0.5
        },
        "color": {
            "inherit": False
        },
        "arrows": {
            "to": {"enabled": True, "scaleFactor": 0.5}
        }
    },
    "interaction": {
        "hover": True,
        "navigationButtons": True
    },
    "layout": {
        "improvedLayout": True
    }
}

In [ ]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_notears.html')

In [199]:
# Remove all outbound edges from center_node
out_edges = list(subgraph.out_edges(center_node))
subgraph.remove_edges_from(out_edges)

In [ ]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_notears.html')

In [201]:
node = "Churn"
edges = []

In [202]:
# Arce care intră în Churn: X -> Churn
for source, target, data in subgraph.in_edges(node, data=True):
    edges.append({
        "Source": source,
        "Target": target,
        "Direction": "IN",
        "Weight": data.get("weight")
    })

In [203]:
df_edges = pd.DataFrame(edges)

df_edges.to_excel(
    "Global_Score_NOTEARS.xlsx",
    index=False
)

In [204]:
from pathlib import Path

In [205]:
center_node = "Churn"
lambdas = [0.25, 0.50, 0.75]
output_file = Path("Path_Based_Structural_Importance_NOTEARS.xlsx")

In [206]:
# ---------------------------------------------------------
# 1. Keep only direct parents of Churn
# ---------------------------------------------------------

parents = list(subgraph.predecessors(center_node))

nodes_to_keep = parents + [center_node]

In [207]:
# Induced subgraph:
# keeps Churn, its direct parents, AND all edges among parents
subgraph = subgraph.subgraph(nodes_to_keep).copy()

print(f"Number of direct parents of {center_node}: {len(parents)}")
print(f"Nodes in subgraph: {subgraph.number_of_nodes()}")
print(f"Edges in subgraph: {subgraph.number_of_edges()}")


Number of direct parents of Churn: 8
Nodes in subgraph: 9
Edges in subgraph: 18


In [223]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_nt_pos_only_parents.html')

supporting_files/Churn_nt_pos_only_parents.html


In [209]:
# ============================================================
# 2. HELPER FUNCTION
# ============================================================

def get_weight(G, source, target):
    return float(G[source][target].get("weight", 0.0))

In [210]:
# ============================================================
# 3. DIRECT COMPONENT D_i
# ============================================================
#
# Signed:
#     D_i = w(X_i -> Churn)
#
# Absolute:
#     |D_i|
#
# ============================================================

D_signed = {}
D_abs = {}

for node in parents:

    w = get_weight(
        subgraph,
        node,
        center_node
    )

    D_signed[node] = w
    D_abs[node] = abs(w)


In [211]:
# ============================================================
# 4. INDIRECT COMPONENT I_i
# ============================================================
#
# Only paths of EXACTLY length 2:
#
#     X_i -> X_j -> Churn
#
# Signed:
#
#     I_i_signed =
#         sum_j w_ij * w_j,Churn
#
# Absolute:
#
#     I_i_abs =
#         sum_j |w_ij * w_j,Churn|
#
# ============================================================

I_signed = {}
I_abs = {}

path_rows = []

for node_i in parents:

    signed_sum = 0.0
    abs_sum = 0.0

    for node_j in parents:

        if node_i == node_j:
            continue

        # Check for:
        #
        # node_i -> node_j -> Churn

        if (
            subgraph.has_edge(node_i, node_j)
            and
            subgraph.has_edge(node_j, center_node)
        ):

            w_ij = get_weight(
                subgraph,
                node_i,
                node_j
            )

            w_j_churn = get_weight(
                subgraph,
                node_j,
                center_node
            )

            path_signed = (
                w_ij * w_j_churn
            )

            path_abs = abs(path_signed)

            signed_sum += path_signed
            abs_sum += path_abs

            path_rows.append({
                "Source": node_i,
                "Via": node_j,
                "Target": center_node,

                "Weight_Source_Via": w_ij,
                "Weight_Via_Churn": w_j_churn,

                "Path_Signed": path_signed,
                "Path_Abs": path_abs
            })

    I_signed[node_i] = signed_sum
    I_abs[node_i] = abs_sum

In [212]:
# ============================================================
# 5. NORMALIZE D_i
# ============================================================
#
# D_abs_norm      in [0, 1]
# D_signed_norm   in [-1, 1]
#
# Both use:
#
#     max |D_i|
#
# ============================================================

max_D_abs = (
    max(D_abs.values())
    if D_abs
    else 0.0
)

D_abs_norm = {}
D_signed_norm = {}

for node in parents:

    if max_D_abs > 0:

        D_abs_norm[node] = (
            D_abs[node] / max_D_abs
        )

        D_signed_norm[node] = (
            D_signed[node] / max_D_abs
        )

    else:

        D_abs_norm[node] = 0.0
        D_signed_norm[node] = 0.0


In [213]:
# ============================================================
# 6. NORMALIZE I_i
# ============================================================
#
# Importance:
#     I_abs_norm in [0,1]
#
# Direction:
#     I_signed_norm in [-1,1]
#
# ============================================================

max_I_abs = (
    max(I_abs.values())
    if I_abs
    else 0.0
)

max_I_signed_abs = (
    max(abs(value) for value in I_signed.values())
    if I_signed
    else 0.0
)


I_abs_norm = {}
I_signed_norm = {}

for node in parents:

    if max_I_abs > 0:

        I_abs_norm[node] = (
            I_abs[node]
            / max_I_abs
        )

    else:

        I_abs_norm[node] = 0.0


    if max_I_signed_abs > 0:

        I_signed_norm[node] = (
            I_signed[node]
            / max_I_signed_abs
        )

    else:

        I_signed_norm[node] = 0.0


In [214]:
# ============================================================
# 7. CALCULATE S_i(lambda) AND C_i(lambda)
# ============================================================
#
# Structural importance:
#
# S_i(lambda) =
#     D_abs_norm
#     +
#     lambda * I_abs_norm
#
#
# Structural direction:
#
# C_i(lambda) =
#     D_signed_norm
#     +
#     lambda * I_signed_norm
#
# ============================================================

rows = []

for node in parents:

    row = {

        "Node": node,

        # Direct component
        "D_signed": D_signed[node],
        "D_abs": D_abs[node],

        "D_signed_norm": D_signed_norm[node],
        "D_abs_norm": D_abs_norm[node],

        # Indirect component
        "I_signed": I_signed[node],
        "I_abs": I_abs[node],

        "I_signed_norm": I_signed_norm[node],
        "I_abs_norm": I_abs_norm[node],
    }


    for lam in lambdas:

        # ----------------------------------------
        # IMPORTANCE
        # ----------------------------------------

        S = (
            D_abs_norm[node]
            +
            lam * I_abs_norm[node]
        )


        # ----------------------------------------
        # DIRECTION
        # ----------------------------------------

        C = (
            D_signed_norm[node]
            +
            lam * I_signed_norm[node]
        )


        row[f"S_{lam:.2f}"] = S
        row[f"C_{lam:.2f}"] = C


        # Direction of recommended intervention
        if C > 0:
            direction = "+"
        elif C < 0:
            direction = "-"
        else:
            direction = "0"

        row[f"Direction_{lam:.2f}"] = direction


    rows.append(row)


results = pd.DataFrame(rows)


In [215]:
# ============================================================
# 8. RANK S_i FOR EACH LAMBDA
# ============================================================
#
# Rank:
#     1 = highest structural importance
#
# Normalized rank:
#
#     1 = best
#     0 = worst
#
# NR = 1 - (Rank - 1)/(N - 1)
#
# ============================================================

n = len(results)

for lam in lambdas:

    S_col = f"S_{lam:.2f}"

    rank_col = f"Rank_{lam:.2f}"

    norm_rank_col = (
        f"Normalized_Rank_{lam:.2f}"
    )


    # Raw rank
    results[rank_col] = (
        results[S_col]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )


    # Normalized rank
    if n > 1:

        results[norm_rank_col] = (
            1
            -
            (
                results[rank_col] - 1
            )
            /
            (n - 1)
        )

    else:

        results[norm_rank_col] = 1.0



In [216]:
# ============================================================
# 9. SORT FOR DISPLAY
# ============================================================
#
# lambda = 0.50 used ONLY for display order.
# It does not affect calculations for other lambdas.
#
# ============================================================

results = (
    results
    .sort_values(
        "Rank_0.50"
    )
    .reset_index(drop=True)
)


In [217]:
# ============================================================
# 10. TABLE WITH INDIVIDUAL LENGTH-2 PATHS
# ============================================================

paths_df = pd.DataFrame(path_rows)

if not paths_df.empty:

    paths_df = (
        paths_df
        .sort_values(
            "Path_Abs",
            ascending=False
        )
        .reset_index(drop=True)
    )


In [218]:
# ============================================================
# 11. TABLE WITH ALL EDGES OF REDUCED SUBGRAPH
# ============================================================

edge_rows = []

for source, target, data in subgraph.edges(data=True):

    edge_rows.append({

        "Source": source,
        "Target": target,

        "Weight": data.get(
            "weight",
            np.nan
        ),

        "Direct_To_Churn":
            target == center_node
    })


edges_df = pd.DataFrame(edge_rows)


In [219]:
# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

display(results)

print("\nLength-2 paths:")
display(paths_df)

print("\nEdges in Churn subgraph:")
display(edges_df)

,Node,D_signed,D_abs,D_signed_norm,D_abs_norm,I_signed,I_abs,I_signed_norm,I_abs_norm,S_0.25,...,Direction_0.50,S_0.75,C_0.75,Direction_0.75,Rank_0.25,Normalized_Rank_0.25,Rank_0.50,Normalized_Rank_0.50,Rank_0.75,Normalized_Rank_0.75
0,MonthlyCharges,0.562001,0.562001,1.000000,1.000000,-0.201528,0.201528,-0.797816,0.797816,1.199454,...,+,1.598362,0.401638,+,1,1.000000,1,1.000000,1,1.000000
1,TotalCharges,-0.390146,0.390146,-0.694209,0.694209,-0.252599,0.252599,-1.000000,1.000000,0.944209,...,-,1.444209,-1.444209,-,2,0.857143,2,0.857143,2,0.857143
2,Tenure,-0.127327,0.127327,-0.226560,0.226560,0.006148,0.130324,0.024340,0.515933,0.355543,...,-,0.613510,-0.208305,-,3,0.714286,3,0.714286,3,0.714286
3,MultipleLines_No_phone_service,0.165818,0.165818,0.295050,0.295050,0.027958,0.027958,0.110682,0.110682,0.322720,...,+,0.378061,0.378061,+,4,0.571429,4,0.571429,4,0.571429
4,TechSupport_No_internet_service,0.086827,0.086827,0.154496,0.154496,0.072389,0.072389,0.286575,0.286575,0.226140,...,+,0.369427,0.369427,+,6,0.285714,5,0.428571,5,0.428571
5,InternetService_No,0.100748,0.100748,0.179267,0.179267,-0.058465,0.058465,-0.231454,0.231454,0.237131,...,+,0.352858,0.005676,+,5,0.428571,6,0.285714,6,0.285714
6,Contract_One_year,-0.098427,0.098427,-0.175137,0.175137,0.000000,0.000000,0.000000,0.000000,0.175137,...,-,0.175137,-0.175137,-,7,0.142857,7,0.142857,7,0.142857
7,TechSupport_No,0.081266,0.081266,0.144601,0.144601,0.000000,0.000000,0.000000,0.000000,0.144601,...,+,0.144601,0.144601,+,8,0.000000,8,0.000000,8,0.000000



Length-2 paths:


,Source,Via,Target,Weight_Source_Via,Weight_Via_Churn,Path_Signed,Path_Abs
0,MonthlyCharges,TotalCharges,Churn,0.516544,-0.390146,-0.201528,0.201528
1,TotalCharges,Tenure,Churn,1.282925,-0.127327,-0.163351,0.163351
2,TotalCharges,MultipleLines_No_phone_service,Churn,-0.470970,0.165818,-0.078096,0.078096
3,TechSupport_No_internet_service,InternetService_No,Churn,0.718509,0.100748,0.072389,0.072389
4,Tenure,MultipleLines_No_phone_service,Churn,0.411512,0.165818,0.068236,0.068236
5,Tenure,Contract_One_year,Churn,0.630801,-0.098427,-0.062088,0.062088
6,InternetService_No,Tenure,Churn,0.234764,-0.127327,-0.029892,0.029892
7,InternetService_No,TechSupport_No,Churn,-0.351605,0.081266,-0.028573,0.028573
8,MultipleLines_No_phone_service,TechSupport_No,Churn,0.344033,0.081266,0.027958,0.027958
9,TotalCharges,Contract_One_year,Churn,0.113309,-0.098427,-0.011153,0.011153



Edges in Churn subgraph:


,Source,Target,Weight,Direct_To_Churn
0,MultipleLines_No_phone_service,TechSupport_No,0.344033,False
1,MultipleLines_No_phone_service,Churn,0.165818,True
2,Tenure,MultipleLines_No_phone_service,0.411512,False
3,Tenure,Contract_One_year,0.630801,False
4,Tenure,Churn,-0.127327,True
5,InternetService_No,Tenure,0.234764,False
6,InternetService_No,TechSupport_No,-0.351605,False
7,InternetService_No,Churn,0.100748,True
8,TechSupport_No,Churn,0.081266,True
9,MonthlyCharges,TotalCharges,0.516544,False


In [220]:
# ============================================================
# 13. EXPORT TO EXCEL
# ============================================================

#output_file = "Path_Based_Churn_Scores.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    results.to_excel(
        writer,
        sheet_name="Scores",
        index=False
    )

    paths_df.to_excel(
        writer,
        sheet_name="Length2_Paths",
        index=False
    )

    edges_df.to_excel(
        writer,
        sheet_name="Subgraph_Edges",
        index=False
    )


In [221]:
print(f"\nResults saved to: {output_file}")


Results saved to: Path_Based_Structural_Importance_NOTEARS.xlsx
